# DEM viewer

The detector error model of a surgery protocol under a chosen noise model, drawn in 3D at the circuit's detector coordinates `(x, y, t)`. Drag to rotate.

- Edges are coloured by the noise channel that causes them. Heralded erasures are always red, drawn as the decoder sees them: herald detectors are removed and each erasure is an ordinary edge.
- Thick edges flip the observable.
- Black is stim's shortest graphlike logical error under all the channels together.

The probabilities only decide edge weights, not which edges exist, so the values in `NOISE` are nominal.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

from dizx.builders import BUILDERS
from dizx.dem_view import dem_mechanisms, graphlike_distance, plot_dem
from dizx.noise_models import NoiseSpec

In [ ]:
# name -> builder; the paper families plus circuit surgery as a no-fusion baseline.
PROTOCOLS = {
    "circuit": "surgery_circuit",
    "bell": "surgery_bell_prep_fusion",
    "bell_zigzag": "surgery_bell_prep_fusion_zigzag",
    "linear_k1": "surgery_hybrid_linear_5chain",
    "linear_k2": "surgery_hybrid_linear_8chain",
    "linear_kinf": "surgery_hybrid_linear_3dchain",
}

p = 1e-3
NOISE = {
    "erasure": {("graph", "fusion_measurement"): NoiseSpec("HERALDED_ERASE", 0.1)},
    "circuit": {
        ("circuit", "reset"): NoiseSpec("RESET_FLIP", p),
        ("circuit", "single_qubit_gate"): NoiseSpec("DEPOLARIZE1", p),
        ("circuit", "two_qubit_gate"): NoiseSpec("DEPOLARIZE2", p),
        ("circuit", "single_qubit_measurement"): NoiseSpec("MEASUREMENT_FLIP", p),
        ("circuit-graph", "fusion_gate"): NoiseSpec("DEPOLARIZE2", p),
        ("circuit-graph", "fusion_measurement"): NoiseSpec("MEASUREMENT_FLIP", p),
    },
    "prep": {
        ("graph", "reset", "matter"): NoiseSpec("RESET_FLIP", p),
        ("graph", "fusion_measurement"): NoiseSpec("MEASUREMENT_FLIP", p),
    },
}


def build(protocol, d, n_cycles=None, observable="ZZ"):
    return BUILDERS[PROTOCOLS[protocol]](distance=d, n_cycles=n_cycles or d, merge_type="ZZ",
                                         observable=observable).circuit

## One protocol

In [ ]:
protocol, d, observable = "bell_zigzag", 3, "XX"
noise = NOISE["erasure"] | NOISE["circuit"]

circuit = build(protocol, d, observable=observable)
fig, ax, mechs = plot_dem(circuit, noise, title=f"{protocol}, d={d}, obs {observable}")

`sources=` limits the drawing to some channels, for example erasure only:

In [ ]:
fig, ax, _ = plot_dem(circuit, noise, sources=["graph/fusion_measurement HERALDED_ERASE"],
                      title=f"{protocol}: erasure edges only")

## The interface as a planar sheet

`sheet=True` draws only the interface: the detectors erasures can flip, flattened onto (position along the seam, t). Circles and squares mark the two sides of the seam. Only mechanisms that lie entirely in the interface are drawn, so the legend reports how much of the shortest logical error falls in the sheet.

In [ ]:
d, observable, noise = 3, "XX", NOISE["erasure"] | NOISE["circuit"]
names = [n for n in PROTOCOLS if n != "circuit"]

fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 7))
for ax, name in zip(axes, names):
    plot_dem(build(name, d, observable=observable), noise, ax=ax, sheet=True, title=name)
fig.tight_layout()

## Protocols side by side

In [ ]:
d, observable, noise = 3, "ZZ", NOISE["erasure"]
names = [n for n in PROTOCOLS if n != "circuit"]

fig = plt.figure(figsize=(14, 9))
for i, name in enumerate(names):
    ax = fig.add_subplot(2, 3, i + 1, projection="3d")
    plot_dem(build(name, d, observable=observable), noise, ax=ax, title=name)
    ax.get_legend().remove()
fig.tight_layout()

## Graphlike distance per protocol and noise model

In [ ]:
d = 3
print(f"{'protocol':<12}{'obs':<5}" + "".join(f"{k:>10}" for k in NOISE))
for name in PROTOCOLS:
    for obs in ("ZZ", "XX"):
        c = build(name, d, observable=obs)
        print(f"{name:<12}{obs:<5}" + "".join(f"{graphlike_distance(c, n) or '-':>10}" for n in NOISE.values()))